# LeetCode 2214 — Minimum Health to Beat Game bằng Python

Link đề: https://leetcode.com/problems/minimum-health-to-beat-game/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Prefix Sum

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def minimumHealth(self, damage: List[int], armor: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển điều kiện "máu luôn **dương** tại mọi thời điểm" thành điều kiện trên **tổng sát thương**: vì máu chỉ giảm, chỉ cần máu **cuối cùng** ≥ 1.
2. Hiểu tác dụng của giáp: dùng tại màn `i` giảm sát thương đi đúng `min(damage[i], armor)` ⇒ nên dùng ở màn có sát thương **lớn nhất**.
3. Rút ra công thức `sum(damage) − min(max(damage), armor) + 1`, và kiểm bằng vét cạn + mô phỏng.
4. Thấy hai góc nhìn khác: **tìm nhị phân theo đáp án** (với hàm kiểm tra đơn điệu) và **DP hai trạng thái** (đã dùng giáp / chưa).
5. Tránh các bẫy: quên `+1`, trừ thẳng `armor` (khi `armor > max`), dùng giáp ở màn sai.

## 2. Đề bài & ví dụ

Trò chơi có `n` màn chơi theo thứ tự `0..n−1`. Qua màn `i` bạn mất `damage[i]` máu. Bạn có một kỹ năng **giáp** dùng **tối đa một lần**, ở một màn bất kỳ: nó chặn tối đa `armor` sát thương của màn đó.

Phải qua các màn **theo thứ tự**, và máu phải **lớn hơn 0 ở mọi thời điểm**. Trả về lượng máu **khởi đầu nhỏ nhất** để thắng.

**Ví dụ 1:** `damage = [2,7,4,3], armor = 4` → `13`. (Dùng giáp ở màn có 4 hoặc 7 sát thương đều chặn được 4.)

**Ví dụ 2:** `damage = [2,5,3,4], armor = 7` → `10`. (Giáp dùng ở màn 5 sát thương chặn hết 5.)

**Ví dụ 3:** `damage = [3,3,3], armor = 0` → `10`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ damage[i] ≤ 10⁵`, `0 ≤ armor ≤ 10⁵`.

In [1]:
import random
import time
from itertools import accumulate, product
from typing import List

examples = [([2, 7, 4, 3], 4, 13), ([2, 5, 3, 4], 7, 10), ([3, 3, 3], 0, 10)]
for d, a, exp in examples:
    print(f"damage={d}, armor={a} -> kỳ vọng {exp}")

damage=[2, 7, 4, 3], armor=4 -> kỳ vọng 13
damage=[2, 5, 3, 4], armor=7 -> kỳ vọng 10
damage=[3, 3, 3], armor=0 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1 Máu chỉ giảm ⇒ chỉ cần kiểm thời điểm cuối

Vì `damage[i] ≥ 0`, máu sau mỗi màn **không tăng**. Máu nhỏ nhất đạt được ở **cuối** game. "Luôn dương" ⇔ "máu cuối ≥ 1" ⇔ `H − tổng_sát_thương_thực ≥ 1` ⇔ `H ≥ tổng + 1`.

In [2]:
H, dmg = 13, [2, 7, 0, 3]               # màn 3 đã dùng giáp chặn 4
health = list(accumulate(dmg, lambda h, x: h - x, initial=H))
print("máu sau từng màn:", health, "| nhỏ nhất:", min(health), "= máu cuối:", health[-1])

máu sau từng màn: [13, 11, 4, 4, 1] | nhỏ nhất: 1 = máu cuối: 1


### 3.2 Lượng giáp chặn được ở một màn

Dùng giáp ở màn `i`: sát thương còn `max(0, damage[i] − armor)`, tức là giảm đúng `min(damage[i], armor)`. Giáp "thừa" (khi `armor > damage[i]`) bị lãng phí.

In [3]:
armor = 7
for d in (2, 5, 9):
    print(f"damage={d}: giáp chặn {min(d, armor)}, còn {max(0, d - armor)}")

damage=2: giáp chặn 2, còn 0
damage=5: giáp chặn 5, còn 0
damage=9: giáp chặn 7, còn 2


### 3.3 `sum`, `max`, `min` trong một lượt

`sum(damage)` và `max(damage)` đều `O(n)`. Muốn chỉ duyệt **một** lần, giữ hai biến chạy.

In [4]:
d = [2, 7, 4, 3]
total = mx = 0
for x in d:
    total += x; mx = max(mx, x)
print(total, mx, "==", sum(d), max(d))

16 7 == 16 7


### 3.4 Tìm nhị phân theo đáp án

Nếu `H` máu đủ thắng thì mọi `H' > H` cũng đủ ⇒ hàm `can(H)` đơn điệu ⇒ tìm `H` nhỏ nhất bằng tìm nhị phân trên `[1, sum + 1]`.

In [5]:
def smallest_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid): hi = mid
        else: lo = mid + 1
    return lo
print(smallest_true(1, 100, lambda h: h * h >= 50))

8


### 3.5 Tràn số `int32` (khi dịch sang Java/C++)

Python có số nguyên không giới hạn, nhưng Java/C++ `int` chỉ chứa tới `2³¹ − 1 ≈ 2,1·10⁹`, còn đáp án bài này tới `10⁵ · 10⁵ + 1 = 10¹⁰ + 1`. Hàm dưới **mô phỏng** phép cộng `int32` (quấn vòng khi tràn) để thấy kết quả sai nếu quên dùng `long`.

In [6]:
def add_int32(x, y):
    s = (x + y) & 0xFFFFFFFF
    return s - (1 << 32) if s >= 1 << 31 else s

total32 = 0
for d in [10**5] * 10**5:
    total32 = add_int32(total32, d)
print("tổng đúng:", 10**10, "| tổng int32 (tràn):", total32)

tổng đúng: 10000000000 | tổng int32 (tràn): 1410065408


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 `damage = [2,7,4,3]`, `armor = 4`. Tổng `16`.

| Dùng giáp ở màn | Giáp chặn | Tổng thực | Máu tối thiểu |
|---|---|---|---|
| không dùng | 0 | 16 | 17 |
| 0 (2) | 2 | 14 | 15 |
| 1 (7) | 4 | 12 | **13** |
| 2 (4) | 4 | 12 | **13** |
| 3 (3) | 3 | 13 | 14 |

Giáp chặn nhiều nhất khi dùng ở màn sát thương **lớn nhất** (chặn `min(max, armor)`). Thời điểm (màn sớm hay muộn) **không quan trọng** vì chỉ máu cuối mới quyết định.

Ví dụ 2: `armor = 7 > max = 5` ⇒ chỉ chặn được 5 ⇒ `14 − 5 + 1 = 10` ✓ (không phải `14 − 7 + 1 = 8`).

In [7]:
d, a = [2, 7, 4, 3], 4
for i in range(-1, len(d)):
    block = 0 if i == -1 else min(d[i], a)
    print(f"giáp ở màn {i if i >= 0 else 'không'}: chặn {block}, cần {sum(d) - block + 1}")

giáp ở màn không: chặn 0, cần 17
giáp ở màn 0: chặn 2, cần 15
giáp ở màn 1: chặn 4, cần 13
giáp ở màn 2: chặn 4, cần 13
giáp ở màn 3: chặn 3, cần 14


**Thời điểm dùng giáp có thật sự không quan trọng?** Hãy kiểm trên một mảng có **hai** màn cùng sát thương lớn nhất ở đầu và cuối: dùng giáp ở màn đầu hay màn cuối, máu tối thiểu như nhau — vì máu nhỏ nhất luôn đạt ở cuối game và tổng sát thương thực như nhau. Điều này **chỉ đúng** khi không có màn hồi máu (mục 15 cho phản ví dụ khi có hồi máu).

In [8]:
d, a = [9, 1, 2, 9], 5
for use in range(len(d)):
    e = [max(0, x - a) if i == use else x for i, x in enumerate(d)]
    print(f"giáp ở màn {use} (damage {d[use]}): sát thương thực {e}, tiền tố {list(accumulate(e))}, cần {sum(e) + 1}")

giáp ở màn 0 (damage 9): sát thương thực [4, 1, 2, 9], tiền tố [4, 5, 7, 16], cần 17
giáp ở màn 1 (damage 1): sát thương thực [9, 0, 2, 9], tiền tố [9, 9, 11, 20], cần 21
giáp ở màn 2 (damage 2): sát thương thực [9, 1, 0, 9], tiền tố [9, 10, 10, 19], cần 20
giáp ở màn 3 (damage 9): sát thương thực [9, 1, 2, 4], tiền tố [9, 10, 12, 16], cần 17


## 5. Cách 1 — Brute Force / baseline: thử dùng giáp ở mỗi màn + mô phỏng

**Ý tưởng:** với mỗi lựa chọn (không dùng / dùng ở màn `i`), **mô phỏng** để tìm máu tối thiểu: máu cần = `1 + max_t (tổng sát thương thực của `t` màn đầu)` — tính bằng cách đi qua các màn. Lấy min trên mọi lựa chọn. Không dùng nhận xét "chỉ máu cuối quan trọng".

In [9]:
def health_brute(damage, armor):
    n = len(damage); best = None
    for use in range(-1, n):                       # -1: không dùng giáp
        lost = worst = 0
        for i, d in enumerate(damage):
            lost += max(0, d - armor) if i == use else d
            worst = max(worst, lost)               # máu phải > lost ở mọi thời điểm
        need = worst + 1
        best = need if best is None or need < best else best
    return best

for d, a, exp in examples:
    got = health_brute(d, a); print(got, "kỳ vọng", exp); assert got == exp

13 kỳ vọng 13
10 kỳ vọng 10
10 kỳ vọng 10


**Độ phức tạp:** `n + 1` lựa chọn × `O(n)` mô phỏng ⇒ `O(n²)`; `n = 10⁵` ⇒ `10¹⁰`, TLE. Space `O(1)`.

**Nút thắt:** ta mô phỏng lại toàn bộ cho mỗi vị trí giáp, trong khi chỉ **một** con số (lượng giáp chặn) thay đổi giữa các lựa chọn.

In [10]:
rng = random.Random(0)
for n in (500, 1000, 2000):
    d = [rng.randint(0, 10**5) for _ in range(n)]
    t = time.perf_counter(); health_brute(d, 50000)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=500: 0.012s
n=1000: 0.046s


n=2000: 0.186s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (máu cuối):** `damage ≥ 0` ⇒ tổng sát thương tiền tố **không giảm** ⇒ `max_t (tiền tố) = tổng`. Vậy máu cần với một lựa chọn giáp = `tổng_thực + 1`.

**Insight 2 (vị trí giáp):** `tổng_thực = sum(damage) − min(damage[i], armor)`; nhỏ nhất khi `damage[i]` lớn nhất ⇒ giảm được `min(max(damage), armor)`. Không dùng giáp không bao giờ tốt hơn (lượng chặn ≥ 0).

$$H_{\min} = \text{sum(damage)} - \min(\max(\text{damage}), \text{armor}) + 1.$$

**Quy tắc "nếu… thì…":**
- **Nếu** `armor ≥ max(damage)` **thì** giáp chặn trọn màn nặng nhất ⇒ trừ `max(damage)`.
- **Ngược lại** giáp bị "bão hòa" ⇒ trừ `armor`.

**Invariant (khi tính một lượt):** sau `i` phần tử, `total` = tổng, `mx` = max của `damage[0..i−1]`.

Hai trường hợp của quy tắc trên, minh họa bằng số: khi `armor` nhỏ hơn màn nặng nhất, giáp **bão hòa** (chặn đúng `armor`); khi `armor` lớn hơn, giáp **thừa** (chặn đúng `max`). Đường cong "lượng chặn theo `armor`" tăng tuyến tính rồi đi ngang tại `max(damage)`.

In [11]:
d = [2, 7, 4, 3]
for a in (0, 2, 4, 6, 7, 9, 20):
    block = min(max(d), a)
    kind = "bão hòa (chặn = armor)" if a < max(d) else "thừa (chặn = max)"
    print(f"armor={a:2d}: chặn {block} -> cần {sum(d) - block + 1:2d}  [{kind}]")

armor= 0: chặn 0 -> cần 17  [bão hòa (chặn = armor)]
armor= 2: chặn 2 -> cần 15  [bão hòa (chặn = armor)]
armor= 4: chặn 4 -> cần 13  [bão hòa (chặn = armor)]
armor= 6: chặn 6 -> cần 11  [bão hòa (chặn = armor)]
armor= 7: chặn 7 -> cần 10  [thừa (chặn = max)]
armor= 9: chặn 7 -> cần 10  [thừa (chặn = max)]
armor=20: chặn 7 -> cần 10  [thừa (chặn = max)]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm nhị phân theo máu khởi đầu

**Giải thích:** `can(H)`: mô phỏng với chiến lược tham lam "dùng giáp ở màn nặng nhất" (hoặc tổng quát: trừ `min(max, armor)`), kiểm máu luôn `> 0`. Tìm `H` nhỏ nhất trong `[1, sum + 1]`. Cách này không cần nhận xét "chỉ máu cuối quan trọng" — hàm `can` mô phỏng đầy đủ từng màn.

In [12]:
def health_bsearch(damage, armor):
    use = max(range(len(damage)), key=damage.__getitem__)     # màn nặng nhất
    def can(H):
        h = H
        for i, d in enumerate(damage):
            h -= max(0, d - armor) if i == use else d
            if h <= 0:
                return False
        return True
    lo, hi = 1, sum(damage) + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if can(mid): hi = mid
        else: lo = mid + 1
    return lo

for d, a, exp in examples:
    assert health_bsearch(d, a) == exp
print([health_bsearch(d, a) for d, a, _ in examples])

[13, 10, 10]


**Độ phức tạp Cách 2:** `O(n · log(sum))` ≈ `10⁵ · 34`. Space `O(1)`.

### 7.2 Cách 3 — DP hai trạng thái (đã dùng giáp / chưa)

`f0` = tổng sát thương tối thiểu tới màn hiện tại **chưa** dùng giáp; `f1` = **đã** dùng (ở màn nào đó trước hoặc tại màn này). Chuyển: `f1 = min(f1 + d, f0 + max(0, d − armor))`, `f0 = f0 + d`. Đáp án `min(f0, f1) + 1`. Đây là khung DP tổng quát khi có "kỹ năng dùng `k` lần" (bài tập 5 dùng ý tương tự).

In [13]:
def health_dp(damage, armor):
    f0, f1 = 0, float("inf")
    for d in damage:
        f0, f1 = f0 + d, min(f1 + d, f0 + max(0, d - armor))
    return min(f0, f1) + 1

for d, a, exp in examples:
    assert health_dp(d, a) == exp
print([health_dp(d, a) for d, a, _ in examples])

[13, 10, 10]


**Độ phức tạp Cách 3:** `O(n)`, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Công thức (bản nộp chính)

In [14]:
def health_formula(damage, armor):
    return sum(damage) - min(max(damage), armor) + 1

for d, a, exp in examples:
    assert health_formula(d, a) == exp
print([health_formula(d, a) for d, a, _ in examples])

[13, 10, 10]


In [15]:
class SolutionOnePass:
    def minimumHealth(self, damage: List[int], armor: int) -> int:
        total = mx = 0
        for d in damage:
            total += d
            if d > mx: mx = d
        return total - min(mx, armor) + 1

print([SolutionOnePass().minimumHealth(d, a) for d, a, _ in examples])

[13, 10, 10]


**Độ phức tạp Cách 4:** `O(n)`, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 (DP) trên ví dụ 2 `damage = [2,5,3,4], armor = 7`:

| i | d | f0 mới = f0 + d | f1 mới = min(f1 + d, f0 + max(0, d−7)) |
|---|---|---|---|
| 0 | 2 | 2 | min(∞, 0 + 0) = 0 |
| 1 | 5 | 7 | min(0 + 5, 2 + 0) = 2 |
| 2 | 3 | 10 | min(2 + 3, 7 + 0) = 5 |
| 3 | 4 | 14 | min(5 + 4, 10 + 0) = 9 |

Đáp án `min(14, 9) + 1 = 10` ✓. Công thức: `14 − min(5, 7) + 1 = 10`.

In [16]:
def trace_dp(damage, armor):
    f0, f1 = 0, float("inf")
    for i, d in enumerate(damage):
        nf0, nf1 = f0 + d, min(f1 + d, f0 + max(0, d - armor))
        print(f"i={i} d={d}: f0 = {f0} + {d} = {nf0} | f1 = min({f1} + {d}, {f0} + {max(0, d - armor)}) = {nf1}")
        f0, f1 = nf0, nf1
    print(f"=> min({f0}, {f1}) + 1 = {min(f0, f1) + 1}")
    return min(f0, f1) + 1

trace_dp([2, 5, 3, 4], 7)

i=0 d=2: f0 = 0 + 2 = 2 | f1 = min(inf + 2, 0 + 0) = 0
i=1 d=5: f0 = 2 + 5 = 7 | f1 = min(0 + 5, 2 + 0) = 2
i=2 d=3: f0 = 7 + 3 = 10 | f1 = min(2 + 3, 7 + 0) = 5
i=3 d=4: f0 = 10 + 4 = 14 | f1 = min(5 + 4, 10 + 0) = 9
=> min(14, 9) + 1 = 10


10

In [17]:
def trace_game(damage, armor, H):
    use = max(range(len(damage)), key=damage.__getitem__); h = H
    print(f"máu khởi đầu {H}, dùng giáp ở màn {use} (damage {damage[use]})")
    for i, d in enumerate(damage):
        lost = max(0, d - armor) if i == use else d; h -= lost
        print(f"  màn {i}: mất {lost}{' (có giáp)' if i == use else ''} -> còn {h}{'  <= 0: THUA' if h <= 0 else ''}")
    return h > 0

trace_game([2, 7, 4, 3], 4, 13)
trace_game([2, 7, 4, 3], 4, 12)

máu khởi đầu 13, dùng giáp ở màn 1 (damage 7)
  màn 0: mất 2 -> còn 11
  màn 1: mất 3 (có giáp) -> còn 8
  màn 2: mất 4 -> còn 4
  màn 3: mất 3 -> còn 1
máu khởi đầu 12, dùng giáp ở màn 1 (damage 7)
  màn 0: mất 2 -> còn 10
  màn 1: mất 3 (có giáp) -> còn 7
  màn 2: mất 4 -> còn 3
  màn 3: mất 3 -> còn 0  <= 0: THUA


False

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def minimumHealth(self, damage: List[int], armor: int) -> int:
        # máu chỉ giảm => chỉ cần máu cuối >= 1
        # giáp dùng ở màn nặng nhất, chặn được min(max(damage), armor)
        return sum(damage) - min(max(damage), armor) + 1

for d, a, exp in examples:
    assert Solution().minimumHealth(d, a) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**(a) Chỉ máu cuối quan trọng.** Với một lựa chọn giáp cố định, sát thương thực ở mỗi màn `e_i ≥ 0`. Máu sau `t` màn là `H − Σ_{i<t} e_i`, không tăng theo `t`. Vậy "máu > 0 mọi lúc" ⇔ `H − Σ e_i > 0` ⇔ `H ≥ Σ e_i + 1`.

**(b) Vị trí giáp tối ưu.** `Σ e_i = sum(damage) − (lượng chặn)`, lượng chặn = `min(damage[j], armor)` nếu dùng ở màn `j`, `0` nếu không dùng. `min(·, armor)` không giảm theo `damage[j]` ⇒ cực đại tại `j = argmax damage`, giá trị `min(max(damage), armor)` (≥ 0 nên luôn nên dùng).

Kết hợp (a), (b): `H_min = sum(damage) − min(max(damage), armor) + 1`.

**Lập luận trao đổi (cách nhìn khác cho (b)).** Giả sử một lời giải tối ưu dùng giáp ở màn `j` không nặng nhất, và `m` là màn nặng nhất. Chuyển giáp từ `j` sang `m`: sát thương thực ở `j` tăng `min(d_j, armor)`, ở `m` giảm `min(d_m, armor) ≥ min(d_j, armor)` ⇒ tổng không tăng ⇒ vẫn tối ưu. Vậy luôn có lời giải tối ưu dùng giáp ở màn nặng nhất.

**DP (Cách 3).** `f0(i)` = tổng sát thương `i` màn đầu không dùng giáp; `f1(i)` = tổng nhỏ nhất `i` màn đầu khi giáp đã dùng ở một màn `< i`. Hoặc giáp đã dùng trước màn `i−1` (`f1(i−1) + d`), hoặc dùng đúng ở màn `i−1` (`f0(i−1) + max(0, d − armor)`).

In [19]:
# Kiểm chứng (a): với MỌI vị trí giáp, máu tối thiểu từ mô phỏng = tổng thực + 1
rng = random.Random(10)
for _ in range(500):
    d = [rng.randint(0, 9) for _ in range(rng.randint(1, 8))]; a = rng.randint(0, 10)
    for use in range(-1, len(d)):
        e = [max(0, x - a) if i == use else x for i, x in enumerate(d)]
        prefix_max = max(accumulate(e))
        assert prefix_max == sum(e)
print("Tiền tố lớn nhất luôn bằng tổng (sát thương không âm) — 500 bộ, mọi vị trí giáp")

Tiền tố lớn nhất luôn bằng tổng (sát thương không âm) — 500 bộ, mọi vị trí giáp


## 11. Phân tích Time / Space

Gọi `n = len(damage)`, `S = sum(damage) ≤ 10¹⁰`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi vị trí + mô phỏng | `O(n²)` | `O(1)` |
| 2. Tìm nhị phân đáp án | `O(n log S)` | `O(1)` |
| 3. DP hai trạng thái | `O(n)` | `O(1)` |
| 4. Công thức | `O(n)` | `O(1)` |

Đáp án tới `10¹⁰ + 1` — vượt `int32` (Java/C++ cần `long`).

In [20]:
rng = random.Random(2214)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for name, fn in [("bsearch", health_bsearch), ("dp", health_dp), ("formula", health_formula),
                 ("one_pass", lambda d, a: SolutionOnePass().minimumHealth(d, a)), ("Solution", lambda d, a: Solution().minimumHealth(d, a))]:
    t = time.perf_counter(); r = [fn(big, a) for a in (0, 50000, 10**5)]
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")
print("toàn 10^5:", Solution().minimumHealth([10**5] * 10**5, 0), "> 2^31 - 1 =", 2**31 - 1)

bsearch  : [4995575594, 4995525594, 4995475594] (0.411s)
dp       : [4995575594, 4995525594, 4995475594] (0.023s)
formula  : [4995575594, 4995525594, 4995475594] (0.004s)
one_pass : [4995575594, 4995525594, 4995475594] (0.008s)
Solution : [4995575594, 4995525594, 4995475594] (0.004s)
toàn 10^5: 10000000001 > 2^31 - 1 = 2147483647


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 1`; mọi sát thương 0 (đáp án 1); `armor = 0`; `armor` lớn hơn mọi sát thương; nhiều màn cùng max.
- **Vét cạn** mọi mảng độ dài `1..5` trên `{0..4}` với mọi `armor ∈ 0..5`, so với brute force mô phỏng.

In [21]:
fns = {"brute": health_brute, "bsearch": health_bsearch, "dp": health_dp, "formula": health_formula,
       "one_pass": lambda d, a: SolutionOnePass().minimumHealth(d, a), "Solution": lambda d, a: Solution().minimumHealth(d, a)}
edge = [([5], 3, 3), ([5], 10, 1), ([0, 0, 0], 4, 1), ([1, 2, 3], 0, 7), ([4, 4, 4], 2, 11), ([10**5], 10**5, 1)]
for d, a, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(d, a) == exp, (name, d, a, fn(d, a))
cnt = 0
for n in range(1, 6):
    for tup in product(range(5), repeat=n):
        for a in range(6):
            want = health_brute(list(tup), a)
            for name, fn in fns.items():
                assert fn(list(tup), a) == want, (name, tup, a)
            cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} bộ: OK")

Ví dụ + biên + vét cạn 23430 bộ: OK


In [22]:
rng = random.Random(42)
for _ in range(300):
    d = [rng.randint(0, 10**5) for _ in range(rng.randint(1, 100))]; a = rng.randint(0, 10**5)
    want = health_brute(d, a)
    for name, fn in fns.items():
        assert fn(d, a) == want, (name, d, a)
print("300 test ngẫu nhiên (giá trị lớn): OK")

300 test ngẫu nhiên (giá trị lớn): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên `+1` (máu phải **lớn hơn** 0)

In [23]:
def wrong_no_plus_one(damage, armor):
    return sum(damage) - min(max(damage), armor)          # SAI: máu cuối = 0 là thua

print("sai:", [wrong_no_plus_one(d, a) for d, a, _ in examples], "| đúng:", [e for _, _, e in examples])

sai: [12, 9, 9] | đúng: [13, 10, 10]


In [24]:
print("sửa:", [Solution().minimumHealth(d, a) for d, a, _ in examples])

sửa: [13, 10, 10]


### Lỗi 2 — Trừ thẳng `armor` (quên giáp bị giới hạn bởi sát thương của màn)

Ví dụ 2: `armor = 7` nhưng màn nặng nhất chỉ 5 ⇒ chỉ chặn được 5.

In [25]:
def wrong_full_armor(damage, armor):
    return sum(damage) - armor + 1                        # SAI

d, a, exp = examples[1]
print("sai:", wrong_full_armor(d, a), "| đúng:", exp)

sai: 8 | đúng: 10


In [26]:
print("sửa (min(max, armor)):", health_formula(d, a))

sửa (min(max, armor)): 10


### Lỗi 3 — Dùng giáp ở màn đầu tiên (hoặc màn cuối) thay vì màn nặng nhất

In [27]:
def wrong_first_level(damage, armor):
    return sum(damage) - min(damage[0], armor) + 1        # SAI: màn 0 chưa chắc nặng nhất

d, a, exp = examples[0]
print("sai:", wrong_first_level(d, a), "| đúng:", exp)

sai: 15 | đúng: 13


In [28]:
print("sửa:", Solution().minimumHealth(d, a))

sửa: 13


### Lỗi 4 — Tìm nhị phân với điều kiện `h < 0` (cho phép máu bằng 0)

Hàm `can` kiểm `h < 0` thay vì `h <= 0` sẽ chấp nhận máu về đúng 0 ⇒ đáp án nhỏ hơn 1 đơn vị.

In [29]:
def wrong_bsearch(damage, armor):
    use = max(range(len(damage)), key=damage.__getitem__)
    def can(H):
        h = H
        for i, d in enumerate(damage):
            h -= max(0, d - armor) if i == use else d
            if h < 0: return False                          # SAI: phải là h <= 0
        return True
    lo, hi = 0, sum(damage) + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if can(mid): hi = mid
        else: lo = mid + 1
    return lo

print("sai:", [wrong_bsearch(d, a) for d, a, _ in examples], "| đúng:", [e for _, _, e in examples])

sai: [12, 9, 9] | đúng: [13, 10, 10]


In [30]:
print("sửa:", [health_bsearch(d, a) for d, a, _ in examples])

sửa: [13, 10, 10]


### Lỗi 5 — DP cập nhật tuần tự (dùng `f0` mới để tính `f1`)

`f1` phải dùng `f0` **cũ** (giáp dùng ở màn hiện tại, khi trước đó chưa dùng).

In [31]:
def wrong_dp_order(damage, armor):
    f0, f1 = 0, float("inf")
    for d in damage:
        f0 = f0 + d                                          # cập nhật trước
        f1 = min(f1 + d, f0 + max(0, d - armor))             # SAI: f0 đã gồm d
    return min(f0, f1) + 1

print("sai:", [wrong_dp_order(d, a) for d, a, _ in examples], "| đúng:", [e for _, _, e in examples])

sai: [17, 15, 10] | đúng: [13, 10, 10]


In [32]:
print("sửa (gán song song):", [health_dp(d, a) for d, a, _ in examples])

sửa (gán song song): [13, 10, 10]


### Lỗi 6 — Chỉ dùng giáp khi `damage[i] ≥ armor` ("để không phí giáp")

Ý nghĩ "chỉ dùng giáp khi chặn được trọn `armor`" khiến ta **không dùng** giáp khi mọi màn đều nhẹ hơn `armor` — trong khi dùng ở màn nặng nhất vẫn chặn được `max(damage)` > 0. Phí một phần giáp vẫn tốt hơn không dùng.

In [33]:
def wrong_only_full_block(damage, armor):
    candidates = [d for d in damage if d >= armor]            # SAI: bỏ các màn nhẹ hơn armor
    block = armor if candidates else 0
    return sum(damage) - block + 1

d, a, exp = examples[1]                                        # armor = 7 > mọi damage
print("sai:", wrong_only_full_block(d, a), "| đúng:", exp)

sai: 15 | đúng: 10


In [34]:
print("sửa (luôn dùng ở màn nặng nhất, chặn min(max, armor)):", Solution().minimumHealth(d, a))

sửa (luôn dùng ở màn nặng nhất, chặn min(max, armor)): 10


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Luôn dương / không âm tại mọi thời điểm"** với các bước chỉ **giảm** ⇒ chỉ cần kiểm **thời điểm cuối**. (Nếu có bước **tăng** — như LeetCode 174 Dungeon Game — thì phải xét tiền tố/DP ngược.)
- **Kỹ năng dùng một lần, giảm tối đa `X`** ⇒ dùng ở phần tử lớn nhất, lợi ích `min(max, X)`.
- Bài tối ưu một lựa chọn rời rạc ⇒ thử mọi lựa chọn (brute) để kiểm công thức.
- Tìm nhị phân theo đáp án là "dự phòng" khi chưa thấy công thức.
- Đáp án lớn ⇒ cẩn thận tràn số ở ngôn ngữ khác.

## 15. Biến thể của bài

1. **Giáp dùng được `k` lần** (bài tập 5): chọn `k` màn nặng nhất, mỗi màn chặn `min(d, armor)`.
2. **Giáp tái sinh / có thời gian hồi** — DP theo trạng thái thời gian hồi.
3. **LeetCode 174 — Dungeon Game (Hard):** có ô hồi máu ⇒ phải DP ngược từ đích.
4. **LeetCode 1964-kiểu "chạy qua chướng ngại"** — kỹ năng dùng một lần: DP hai trạng thái như Cách 3.
5. **Có màn hồi máu** (`damage[i] < 0` cho phép): không còn "chỉ máu cuối quan trọng"; phải xét max tiền tố.

Minh họa biến thể 5 (tự dựng, kiểm bằng mô phỏng): với màn hồi máu, máu cần = `1 + max tiền tố` và vị trí giáp ảnh hưởng tới **tiền tố lớn nhất**, không chỉ tổng.

In [35]:
def health_with_heal(damage, armor):
    # damage có thể âm (hồi máu). Thử mọi vị trí giáp, tính 1 + max(0, max tiền tố)
    best = None
    for use in range(-1, len(damage)):
        lost = worst = 0
        for i, d in enumerate(damage):
            lost += (d - min(d, armor) if d > 0 else d) if i == use else d
            worst = max(worst, lost)
        best = worst + 1 if best is None or worst + 1 < best else best
    return best

print("có hồi máu [5, -10, 3, 3, 3], armor=3:", health_with_heal([5, -10, 3, 3, 3], 3),
      "| công thức bài gốc (sai ở đây):", sum([5, -10, 3, 3, 3]) - min(5, 3) + 1)

có hồi máu [5, -10, 3, 3, 3], armor=3: 3 | công thức bài gốc (sai ở đây): 2


Minh họa biến thể 3 — **LeetCode 174 Dungeon Game** (tự dựng dữ liệu nhỏ, kiểm bằng vét cạn mọi đường đi). Khi có ô **hồi máu**, "máu cuối" không còn quyết định; ta làm DP **ngược** từ ô đích: `need[i][j]` = máu tối thiểu khi **bước vào** ô `(i, j)` để tới được đích: `need = max(1, min(need_phải, need_dưới) − grid[i][j])`.

In [36]:
def dungeon_dp(grid):
    m, n = len(grid), len(grid[0]); INF = float("inf")
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - grid[i][j])
    return need[0][0]

def dungeon_brute(grid):
    m, n = len(grid), len(grid[0]); best = None
    for moves in product("RD", repeat=m + n - 2):
        if moves.count("D") != m - 1: continue
        i = j = 0; s = grid[0][0]; worst = s
        for mv in moves:
            if mv == "D": i += 1
            else: j += 1
            s += grid[i][j]; worst = min(worst, s)
        need = max(1, 1 - worst)
        best = need if best is None or need < best else best
    return best

rng = random.Random(174)
for _ in range(200):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    g = [[rng.randint(-8, 5) for _ in range(n)] for _ in range(m)]
    assert dungeon_dp(g) == dungeon_brute(g)
print("Dungeon Game (DP ngược) khớp vét cạn; ví dụ LeetCode ->", dungeon_dp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]))

Dungeon Game (DP ngược) khớp vét cạn; ví dụ LeetCode -> 7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi vị trí giáp | mô phỏng từng lựa chọn | `O(n²)` | `O(1)` | kiểm tra |
| 2. Tìm nhị phân đáp án | `can(H)` đơn điệu | `O(n log S)` | `O(1)` | khi chưa có công thức |
| 3. DP hai trạng thái | đã/chưa dùng giáp | `O(n)` | `O(1)` | tổng quát hóa (k lần, hồi máu) |
| 4. Công thức | `sum − min(max, armor) + 1` | `O(n)` | `O(1)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `minimumHealth([1, 1, 1, 1], 10)`.
2. **Tính tay:** `minimumHealth([8, 2, 6], 5)`.
3. **Điền TODO:** `best_armor_level(damage, armor)` trả về chỉ số màn nên dùng giáp (màn nặng nhất đầu tiên).
4. **Tự viết lại:** viết lại công thức một lượt (không gọi `sum`/`max`).
5. **Mini challenge:** `minimum_health_k(damage, armor, k)` — giáp dùng được tối đa `k` lần (mỗi màn tối đa một lần). Kiểm bằng vét cạn mọi tập `≤ k` màn.

In [37]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_armor_level(damage, armor):
    # TODO
    pass

In [38]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_minimum_health(damage, armor):
    # TODO: total, mx trong một vòng lặp
    pass

In [39]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def minimum_health_k(damage, armor, k):
    # TODO: sắp giảm dần, lấy k phần tử lớn nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Tổng 4, max 1. (2) Giáp chặn `min(1, 10) = 1`. (3) `4 − 1 + 1 = 4`.

**Bài 2.** (1) Tổng 16, max 8. (2) Chặn `min(8, 5) = 5`. (3) `16 − 5 + 1 = 12`.

**Bài 3.** (1) `max(range(n), key=damage.__getitem__)`. (2) `max` trả chỉ số đầu tiên khi hòa. (3) Mọi màn có `damage = max` đều tối ưu như nhau.

**Bài 4.** (1) `total = mx = 0`. (2) Cập nhật trong vòng `for`. (3) `total − min(mx, armor) + 1`.

**Bài 5.** (1) Lợi ích dùng ở màn `i` là `min(d_i, armor)` — độc lập giữa các màn. (2) Chọn `k` màn có lợi ích lớn nhất = `k` màn nặng nhất. (3) `sum − Σ min(top_k, armor) + 1`.

## 19. Đáp án tham khảo

In [40]:
# Bài 1, 2
assert Solution().minimumHealth([1, 1, 1, 1], 10) == 4
assert Solution().minimumHealth([8, 2, 6], 5) == 12
print("Bài 1: 4 | Bài 2: 12")
trace_game([8, 2, 6], 5, 12); trace_game([8, 2, 6], 5, 11)

Bài 1: 4 | Bài 2: 12
máu khởi đầu 12, dùng giáp ở màn 0 (damage 8)
  màn 0: mất 3 (có giáp) -> còn 9
  màn 1: mất 2 -> còn 7
  màn 2: mất 6 -> còn 1
máu khởi đầu 11, dùng giáp ở màn 0 (damage 8)
  màn 0: mất 3 (có giáp) -> còn 8
  màn 1: mất 2 -> còn 6
  màn 2: mất 6 -> còn 0  <= 0: THUA


False

In [41]:
# Bài 3, 4
def best_armor_level_ans(damage, armor):
    return max(range(len(damage)), key=damage.__getitem__)

def my_minimum_health_ans(damage, armor):
    total = mx = 0
    for d in damage:
        total += d; mx = d if d > mx else mx
    return total - min(mx, armor) + 1

rng = random.Random(3)
for _ in range(300):
    d = [rng.randint(0, 20) for _ in range(rng.randint(1, 10))]; a = rng.randint(0, 25)
    i = best_armor_level_ans(d, a)
    assert sum(d) - min(d[i], a) + 1 == health_brute(d, a) == my_minimum_health_ans(d, a)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [42]:
# Bài 5
from itertools import combinations
def minimum_health_k_ans(damage, armor, k):
    top = sorted(damage, reverse=True)[:k]
    return sum(damage) - sum(min(d, armor) for d in top) + 1

def minimum_health_k_brute(damage, armor, k):
    n = len(damage); best = None
    for r in range(0, min(k, n) + 1):
        for S in combinations(range(n), r):
            need = sum(max(0, d - armor) if i in S else d for i, d in enumerate(damage)) + 1
            best = need if best is None or need < best else best
    return best

rng = random.Random(5)
for _ in range(300):
    d = [rng.randint(0, 15) for _ in range(rng.randint(1, 7))]; a = rng.randint(0, 12); k = rng.randint(0, 4)
    assert minimum_health_k_ans(d, a, k) == minimum_health_k_brute(d, a, k)
assert all(minimum_health_k_ans(d, a, 1) == Solution().minimumHealth(d, a) for d, a, _ in examples)
print("Bài 5: OK; ví dụ 1 với k = 2 ->", minimum_health_k_ans([2, 7, 4, 3], 4, 2))

Bài 5: OK; ví dụ 1 với k = 2 -> 9


**Giải thích thêm cho Bài 5 (giáp dùng `k` lần).** Lợi ích dùng giáp ở màn `i` là `g_i = min(d_i, armor)` và các lợi ích **độc lập** (dùng ở màn này không ảnh hưởng màn khác). Tổng lợi ích của tập `S` (`|S| ≤ k`) là `Σ_{i∈S} g_i` ⇒ lớn nhất khi lấy `k` giá trị `g_i` lớn nhất. Vì `g_i` không giảm theo `d_i`, `k` màn nặng nhất cho `k` lợi ích lớn nhất. Bảng dưới in máu cần theo `k` cho ví dụ 1: lợi ích giảm dần rồi bão hòa khi hết màn có ích.

In [43]:
d, a = [2, 7, 4, 3], 4
for k in range(0, len(d) + 1):
    top = sorted(d, reverse=True)[:k]
    print(f"k={k}: dùng giáp ở các màn {top}, chặn {[min(x, a) for x in top]} -> cần {minimum_health_k_ans(d, a, k)}")

k=0: dùng giáp ở các màn [], chặn [] -> cần 17
k=1: dùng giáp ở các màn [7], chặn [4] -> cần 13
k=2: dùng giáp ở các màn [7, 4], chặn [4, 4] -> cần 9
k=3: dùng giáp ở các màn [7, 4, 3], chặn [4, 4, 3] -> cần 6
k=4: dùng giáp ở các màn [7, 4, 3, 2], chặn [4, 4, 3, 2] -> cần 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
return sum(damage) - min(max(damage), armor) + 1
```

**Template DP "kỹ năng dùng một lần":**

```python
f0, f1 = 0, inf                 # chưa dùng / đã dùng
for d in damage:
    f0, f1 = f0 + d, min(f1 + d, f0 + max(0, d - armor))
return min(f0, f1) + 1
```

In [44]:
def template(damage, armor):
    return sum(damage) - min(max(damage), armor) + 1
print([template(d, a) for d, a, _ in examples])

[13, 10, 10]


**Checklist:**

- [ ] `+1` vì máu phải **lớn hơn** 0.
- [ ] Giáp chặn `min(max(damage), armor)` (không trừ thẳng `armor`).
- [ ] Giáp dùng ở màn **nặng nhất** (vị trí sớm/muộn không quan trọng).
- [ ] Nếu tìm nhị phân: điều kiện thua là `h <= 0`.
- [ ] Nếu DP: gán song song `f0, f1`.
- [ ] Đáp án có thể > `2³¹ − 1`.